In [1]:
import numpy as np
import time
import os

print(f"Logical CPU Cores Detected: {os.cpu_count()}")
print("Generating 50,000 samples with 1,000 features in RAM...")

np.random.seed(42)

# Dataset: 50,000 samples, 1,000 features (~400 MB in 64-bit float)
N_SAMPLES = 50000
N_FEATURES = 1000
N_HIDDEN_1 = 512
N_HIDDEN_2 = 256
N_CLASSES = 10

X = np.random.randn(N_SAMPLES, N_FEATURES)
# Create synthetic ground-truth labels (One-Hot Encoded)
y_indices = np.random.randint(0, N_CLASSES, size=N_SAMPLES)
Y = np.eye(N_CLASSES)[y_indices]

# Initialize network weights
W1 = np.random.randn(N_FEATURES, N_HIDDEN_1) * np.sqrt(2.0 / N_FEATURES)
b1 = np.zeros((1, N_HIDDEN_1))

W2 = np.random.randn(N_HIDDEN_1, N_HIDDEN_2) * np.sqrt(2.0 / N_HIDDEN_1)
b2 = np.zeros((1, N_HIDDEN_2))

W3 = np.random.randn(N_HIDDEN_2, N_CLASSES) * np.sqrt(2.0 / N_HIDDEN_2)
b3 = np.zeros((1, N_CLASSES))

learning_rate = 0.05
epochs = 60  # Increase to 100 if your CPU finishes this under 60 seconds

print(f"\nStarting heavy matrix training for {epochs} epochs...")
print("-" * 65)
start_total = time.perf_counter()

for epoch in range(1, epochs + 1):
    epoch_start = time.perf_counter()
    
    # --- 1. Forward Propagation (Heavy Matrix Multiplications) ---
    Z1 = X @ W1 + b1
    A1 = np.maximum(0, Z1)  # ReLU activation
    
    Z2 = A1 @ W2 + b2
    A2 = np.maximum(0, Z2)  # ReLU activation
    
    Z3 = A2 @ W3 + b3
    # Softmax activation
    exp_scores = np.exp(Z3 - np.max(Z3, axis=1, keepdims=True))
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    
    # Cross-entropy loss
    loss = -np.sum(Y * np.log(probs + 1e-8)) / N_SAMPLES
    
    # --- 2. Backward Propagation (Gradient Computation) ---
    dZ3 = (probs - Y) / N_SAMPLES
    dW3 = A2.T @ dZ3
    db3 = np.sum(dZ3, axis=0, keepdims=True)
    
    dA2 = dZ3 @ W3.T
    dZ2 = dA2 * (Z2 > 0)
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)
    
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (Z1 > 0)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    
    # --- 3. Parameter Updates ---
    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    
    epoch_time = time.perf_counter() - epoch_start
    
    if epoch % 5 == 0 or epoch == 1:
        elapsed = time.perf_counter() - start_total
        print(f"Epoch {epoch:3d}/{epochs} | Loss: {loss:.4f} | "
              f"Step Time: {epoch_time:.2f}s | Total Elapsed: {elapsed:.1f}s")

total_duration = time.perf_counter() - start_total
print("-" * 65)
print(f"Benchmark Complete!")
print(f"Total CPU Execution Time : {total_duration:.2f} seconds ({total_duration/60:.2f} minutes)")
print(f"Average Time per Epoch   : {total_duration/epochs:.2f} seconds")

Logical CPU Cores Detected: 12
Generating 50,000 samples with 1,000 features in RAM...

Starting heavy matrix training for 60 epochs...
-----------------------------------------------------------------
Epoch   1/60 | Loss: 3.2764 | Step Time: 3.42s | Total Elapsed: 3.4s
Epoch   5/60 | Loss: 2.7003 | Step Time: 2.69s | Total Elapsed: 14.5s
Epoch  10/60 | Loss: 2.6523 | Step Time: 2.59s | Total Elapsed: 27.9s
Epoch  15/60 | Loss: 2.6121 | Step Time: 2.30s | Total Elapsed: 40.1s
Epoch  20/60 | Loss: 2.5781 | Step Time: 2.07s | Total Elapsed: 50.8s
Epoch  25/60 | Loss: 2.5492 | Step Time: 2.11s | Total Elapsed: 61.3s
Epoch  30/60 | Loss: 2.5243 | Step Time: 2.21s | Total Elapsed: 72.3s
Epoch  35/60 | Loss: 2.5028 | Step Time: 2.19s | Total Elapsed: 83.4s
Epoch  40/60 | Loss: 2.4841 | Step Time: 2.30s | Total Elapsed: 94.6s
Epoch  45/60 | Loss: 2.4675 | Step Time: 2.27s | Total Elapsed: 106.3s
Epoch  50/60 | Loss: 2.4528 | Step Time: 2.18s | Total Elapsed: 117.8s
Epoch  55/60 | Loss: 2.4397

In [1]:
import cupy as cp
import time

# Detect and display the connected NVIDIA GPU
gpu_info = cp.cuda.runtime.getDeviceProperties(0)
print(f"GPU Connected: {gpu_info['name'].decode('utf-8')}")
print("Generating 50,000 samples with 1,000 features directly in GPU VRAM...")

cp.random.seed(42)

# Identical architecture and dimensions (64-bit float to match CPU test)
N_SAMPLES = 50000
N_FEATURES = 1000
N_HIDDEN_1 = 512
N_HIDDEN_2 = 256
N_CLASSES = 10

X = cp.random.randn(N_SAMPLES, N_FEATURES, dtype=cp.float64)
y_indices = cp.random.randint(0, N_CLASSES, size=N_SAMPLES)
Y = cp.eye(N_CLASSES, dtype=cp.float64)[y_indices]

W1 = cp.random.randn(N_FEATURES, N_HIDDEN_1, dtype=cp.float64) * cp.sqrt(2.0 / N_FEATURES)
b1 = cp.zeros((1, N_HIDDEN_1), dtype=cp.float64)

W2 = cp.random.randn(N_HIDDEN_1, N_HIDDEN_2, dtype=cp.float64) * cp.sqrt(2.0 / N_HIDDEN_1)
b2 = cp.zeros((1, N_HIDDEN_2), dtype=cp.float64)

W3 = cp.random.randn(N_HIDDEN_2, N_CLASSES, dtype=cp.float64) * cp.sqrt(2.0 / N_HIDDEN_2)
b3 = cp.zeros((1, N_CLASSES), dtype=cp.float64)

learning_rate = 0.05
epochs = 60

# Warm up GPU CUDA kernels before starting the clock
cp.cuda.Stream.null.synchronize()

print(f"\nStarting GPU matrix training for {epochs} epochs...")
print("-" * 65)
start_total = time.perf_counter()

for epoch in range(1, epochs + 1):
    epoch_start = time.perf_counter()
    
    # --- 1. Forward Propagation ---
    Z1 = X @ W1 + b1
    A1 = cp.maximum(0, Z1)
    
    Z2 = A1 @ W2 + b2
    A2 = cp.maximum(0, Z2)
    
    Z3 = A2 @ W3 + b3
    exp_scores = cp.exp(Z3 - cp.max(Z3, axis=1, keepdims=True))
    probs = exp_scores / cp.sum(exp_scores, axis=1, keepdims=True)
    
    loss = -cp.sum(Y * cp.log(probs + 1e-8)) / N_SAMPLES
    
    # --- 2. Backward Propagation ---
    dZ3 = (probs - Y) / N_SAMPLES
    dW3 = A2.T @ dZ3
    db3 = cp.sum(dZ3, axis=0, keepdims=True)
    
    dA2 = dZ3 @ W3.T
    dZ2 = dA2 * (Z2 > 0)
    dW2 = A1.T @ dZ2
    db2 = cp.sum(dZ2, axis=0, keepdims=True)
    
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (Z1 > 0)
    dW1 = X.T @ dZ1
    db1 = cp.sum(dZ1, axis=0, keepdims=True)
    
    # --- 3. Parameter Updates ---
    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    
    # Wait for GPU to finish calculations before recording exact time
    cp.cuda.Stream.null.synchronize()
    epoch_time = time.perf_counter() - epoch_start
    
    if epoch % 5 == 0 or epoch == 1:
        elapsed = time.perf_counter() - start_total
        print(f"Epoch {epoch:3d}/{epochs} | Loss: {float(loss):.4f} | "
              f"Step Time: {epoch_time:.4f}s | Total Elapsed: {elapsed:.2f}s")

total_duration = time.perf_counter() - start_total
cpu_baseline = 139.72

print("-" * 65)
print(f"GPU Benchmark Complete!")
print(f"Total GPU Execution Time : {total_duration:.2f} seconds")
print(f"Average Time per Epoch   : {total_duration/epochs:.4f} seconds")
print(f"Speedup vs Local i3-1220P: {cpu_baseline / total_duration:.1f}x FASTER")

GPU Connected: Tesla T4
Generating 50,000 samples with 1,000 features directly in GPU VRAM...

Starting GPU matrix training for 60 epochs...
-----------------------------------------------------------------
Epoch   1/60 | Loss: 2.8882 | Step Time: 2.4574s | Total Elapsed: 2.46s
Epoch   5/60 | Loss: 2.6943 | Step Time: 0.6233s | Total Elapsed: 4.95s
Epoch  10/60 | Loss: 2.6474 | Step Time: 0.6570s | Total Elapsed: 8.14s
Epoch  15/60 | Loss: 2.6080 | Step Time: 0.6571s | Total Elapsed: 11.42s
Epoch  20/60 | Loss: 2.5747 | Step Time: 0.6565s | Total Elapsed: 14.71s
Epoch  25/60 | Loss: 2.5462 | Step Time: 0.6565s | Total Elapsed: 17.99s
Epoch  30/60 | Loss: 2.5217 | Step Time: 0.6569s | Total Elapsed: 21.28s
Epoch  35/60 | Loss: 2.5005 | Step Time: 0.6571s | Total Elapsed: 24.56s
Epoch  40/60 | Loss: 2.4820 | Step Time: 0.6568s | Total Elapsed: 27.84s
Epoch  45/60 | Loss: 2.4657 | Step Time: 0.6567s | Total Elapsed: 31.13s
Epoch  50/60 | Loss: 2.4512 | Step Time: 0.6566s | Total Elapsed: 

In [2]:
import cupy as cp
import time

# Same 50,000 samples x 1,000 features, now in standard GPU 32-bit float (FP32)
cp.random.seed(42)
N_SAMPLES, N_FEATURES = 50000, 1000
N_HIDDEN_1, N_HIDDEN_2, N_CLASSES = 512, 256, 10

X = cp.random.randn(N_SAMPLES, N_FEATURES, dtype=cp.float32)
y_indices = cp.random.randint(0, N_CLASSES, size=N_SAMPLES)
Y = cp.eye(N_CLASSES, dtype=cp.float32)[y_indices]

W1 = cp.random.randn(N_FEATURES, N_HIDDEN_1, dtype=cp.float32) * cp.sqrt(2.0 / N_FEATURES, dtype=cp.float32)
b1 = cp.zeros((1, N_HIDDEN_1), dtype=cp.float32)
W2 = cp.random.randn(N_HIDDEN_1, N_HIDDEN_2, dtype=cp.float32) * cp.sqrt(2.0 / N_HIDDEN_1, dtype=cp.float32)
b2 = cp.zeros((1, N_HIDDEN_2), dtype=cp.float32)
W3 = cp.random.randn(N_HIDDEN_2, N_CLASSES, dtype=cp.float32) * cp.sqrt(2.0 / N_HIDDEN_2, dtype=cp.float32)
b3 = cp.zeros((1, N_CLASSES), dtype=cp.float32)

learning_rate = cp.float32(0.05)
epochs = 60

cp.cuda.Stream.null.synchronize()
start_total = time.perf_counter()

for epoch in range(1, epochs + 1):
    epoch_start = time.perf_counter()
    
    # Forward Pass
    Z1 = X @ W1 + b1
    A1 = cp.maximum(0, Z1)
    Z2 = A1 @ W2 + b2
    A2 = cp.maximum(0, Z2)
    Z3 = A2 @ W3 + b3
    
    exp_scores = cp.exp(Z3 - cp.max(Z3, axis=1, keepdims=True))
    probs = exp_scores / cp.sum(exp_scores, axis=1, keepdims=True)
    loss = -cp.sum(Y * cp.log(probs + 1e-8)) / N_SAMPLES
    
    # Backward Pass
    dZ3 = (probs - Y) / N_SAMPLES
    dW3 = A2.T @ dZ3
    db3 = cp.sum(dZ3, axis=0, keepdims=True)
    dA2 = dZ3 @ W3.T
    dZ2 = dA2 * (Z2 > 0)
    dW2 = A1.T @ dZ2
    db2 = cp.sum(dZ2, axis=0, keepdims=True)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (Z1 > 0)
    dW1 = X.T @ dZ1
    db1 = cp.sum(dZ1, axis=0, keepdims=True)
    
    # Updates
    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    
    cp.cuda.Stream.null.synchronize()
    epoch_time = time.perf_counter() - epoch_start
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d}/60 | Loss: {float(loss):.4f} | Step Time: {epoch_time:.4f}s")

total_duration = time.perf_counter() - start_total
print("-" * 60)
print(f"FP32 GPU Execution Time  : {total_duration:.2f} seconds!")
print(f"Speedup vs Local i3-1220P: {139.72 / total_duration:.1f}x FASTER")

Epoch  1/60 | Loss: 3.0137 | Step Time: 1.9406s
Epoch 10/60 | Loss: 2.6285 | Step Time: 0.0412s
Epoch 20/60 | Loss: 2.5604 | Step Time: 0.0401s
Epoch 30/60 | Loss: 2.5108 | Step Time: 0.0404s
Epoch 40/60 | Loss: 2.4735 | Step Time: 0.0405s
Epoch 50/60 | Loss: 2.4445 | Step Time: 0.0404s
Epoch 60/60 | Loss: 2.4215 | Step Time: 0.0404s
------------------------------------------------------------
FP32 GPU Execution Time  : 4.41 seconds!
Speedup vs Local i3-1220P: 31.7x FASTER
